# Model Experiments
**Walmart Sales Forecasting — Model Comparison Notebook**

Use this notebook to run ad-hoc experiments, tune hyperparameters,
and compare model outputs visually before committing to `main.py`.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from data.data_loader import load_config, load_processed_data, load_features_data
from evaluation.metrics import evaluate_model, compare_models
from evaluation.backtesting import simple_time_split, get_feature_columns

config = load_config('../config/config.yaml')
df_raw = load_processed_data(config)

# Pick a single Store-Dept for quick experiments
STORE, DEPT = 1, 1
series = df_raw[(df_raw['Store'] == STORE) & (df_raw['Dept'] == DEPT)].set_index('Date')['Weekly_Sales'].sort_index()
print(f'Series length: {len(series)} weeks')
series.plot(title=f'Store {STORE} Dept {DEPT} — Weekly Sales', figsize=(14, 4))
plt.show()

## ARIMA vs Prophet Side-by-Side

In [ ]:
from models.arima_model import ARIMAForecaster
from models.prophet_model import ProphetForecaster

split = int(len(series) * 0.8)
train, test = series.iloc[:split], series.iloc[split:]

results = []

# ARIMA
arima = ARIMAForecaster(config)
arima.fit(train)
arima_preds = arima.predict(steps=len(test))
results.append(evaluate_model(test.values, arima_preds, 'ARIMA'))

# Prophet
prophet = ProphetForecaster(config)
prophet.fit(train)
prophet_preds = prophet.predict_in_sample(test)
results.append(evaluate_model(test.values, prophet_preds, 'Prophet'))

# Plot
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(test.index, test.values, label='Actual', color='#1f77b4', linewidth=2)
ax.plot(test.index, arima_preds, label='ARIMA', color='#ff7f0e', linestyle='--')
ax.plot(test.index, prophet_preds, label='Prophet', color='#2ca02c', linestyle='-.')
ax.set_title(f'Store {STORE} Dept {DEPT} — ARIMA vs Prophet')
ax.legend(); plt.tight_layout(); plt.show()

compare_models(results)

## XGBoost with Feature Engineering

In [ ]:
from features.feature_engineering import engineer_features
from models.regression_model import XGBoostModel

features_df = engineer_features(df_raw, config)
sample = features_df[(features_df['Store'] == STORE) & (features_df['Dept'] == DEPT)].copy()
feature_cols = get_feature_columns(sample)
sample = sample.dropna(subset=feature_cols)

X_train, X_test, y_train, y_test = simple_time_split(sample, 'Weekly_Sales', feature_cols)

xgb = XGBoostModel(config)
xgb.fit(X_train, y_train)
xgb_preds = xgb.predict(X_test)
results.append(evaluate_model(y_test, xgb_preds, 'XGBoost'))

# Feature importance
importances = xgb.feature_importances(feature_cols)
importances.head(15).plot(kind='barh', figsize=(10, 6), color='steelblue')
plt.title('Top 15 Feature Importances — XGBoost')
plt.gca().invert_yaxis()
plt.tight_layout(); plt.show()